In [55]:
project_root = "/home/krzysztof/studia/magisterka/time-series-invariance"

In [56]:
import os
subdir = ""
os.chdir(os.path.join(project_root, subdir))

import numpy as np

from src.data_generation.simple_data_generation import shift_time_series, shrunk_time_series
from notebooks.evaluations.helpers import (
        apply_embedding_function_shrunk,
        apply_embedding_function_shifts,
        load_dataset,
        load_distortions,
    )

In [57]:
subdir = "TS-TCC"
os.chdir(os.path.join(project_root, subdir))

In [58]:
import torch
from models.model import base_Model
from models.TC import TC
import numpy as np

device = torch.device('cpu')

In [59]:
import importlib


def load_models(data_type, device, training_type="fine_tune"):
    exec(f"from config_files.{data_type}_Configs import Config as Configs")
    module_name = f"config_files.{data_type}_Configs"
    module = importlib.import_module(module_name)
    importlib.reload(module)

    configs = module.Config()

    model = base_Model(configs).to(device)
    temporal_contr_model = TC(configs, device).to(device)

    load_from = f"experiments_logs/exp_{data_type}/run_timestamp/{training_type}_seed_123/saved_models"

    chkpoint = torch.load(os.path.join(load_from, "ckp_last.pt"), map_location=device)

    model.load_state_dict(chkpoint["model_state_dict"])
    temporal_contr_model.load_state_dict(chkpoint["temporal_contr_model_state_dict"])

    return model, temporal_contr_model


In [60]:
def ts_tcc_embedding_function(X, model, temporal_contr_model, device):
    if X.ndim == 3:
        X = X.transpose(1, 2)
    if X.ndim == 1:
        X = X[None, None, :]
    if X.ndim == 2:
        X = X[:, None, :]
    _, features = model(torch.from_numpy(X).float().to(device))
    z_aug2 = features.transpose(1, 2)
    embedding = temporal_contr_model.seq_transformer(z_aug2)

    return embedding.detach().cpu().numpy()[0]

## Run experiments

In [61]:
data_dir = "../data/datasets"
dataset_names = [name[:-4] for name in os.listdir(data_dir)]
dataset_names

['CBF',
 'OSULeaf',
 'Adiac',
 'Fish',
 'MedicalImages',
 'Wafer',
 'SwedishLeaf',
 'FacesUCR',
 'SyntheticControl',
 'FaceFour',
 'Trace',
 'Beef',
 'GunPoint',
 'TwoPatterns',
 'Coffee',
 'OliveOil',
 'ECG200']

### Fine tune

In [62]:
model_dict = {}

for dataset_name in dataset_names:
    model, temporal_contr_model = load_models(dataset_name, device)
    model_dict[dataset_name] = (model, temporal_contr_model)

In [63]:
subdir = ""
os.chdir(os.path.join(project_root, subdir))

In [64]:
model, temporal_contr_model = model_dict['Adiac']
model.embedding_only = True
model.num_classes = 3

data = load_dataset(dataset_name)
X = data
X = X[:, None, :]
print(X.shape)

_, features = model(torch.from_numpy(X).float().to(device))
z_aug2 = features.transpose(1, 2)
embedding = temporal_contr_model.seq_transformer(z_aug2)

embedding.shape

(200, 1, 96)


torch.Size([200, 100])

In [65]:
distortion_type = "shrink"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    print(f"data shape: {data.shape}")
    random_shrinks = load_distortions(dataset_name, distortion_type)

    model, temporal_contr_model = model_dict[dataset_name]
    
    embeddings = apply_embedding_function_shrunk(data, lambda x: ts_tcc_embedding_function(x, model, temporal_contr_model, device), random_shrinks)
    print(f"embeddings shape: {embeddings.shape}")
    path = f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/{distortion_type}/{dataset_name}.npy"
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (442, 427)
embeddings shape: (442, 6, 100)
data shape: (500, 176)
embeddings shape: (500, 6, 100)
data shape: (350, 463)
embeddings shape: (350, 6, 100)
data shape: (500, 99)
embeddings shape: (500, 6, 100)
data shape: (500, 152)
embeddings shape: (500, 6, 100)
data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (500, 131)
embeddings shape: (500, 6, 100)
data shape: (500, 60)
embeddings shape: (500, 6, 100)
data shape: (112, 350)
embeddings shape: (112, 6, 100)
data shape: (200, 275)
embeddings shape: (200, 6, 100)
data shape: (60, 470)
embeddings shape: (60, 6, 100)
data shape: (200, 150)
embeddings shape: (200, 6, 100)
data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (56, 286)
embeddings shape: (56, 6, 100)
data shape: (60, 570)
embeddings shape: (60, 6, 100)
data shape: (200, 96)
embeddings shape: (200, 6, 100)


In [66]:
distortion_type = "shift"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    data = data[:, :, None]
    print(f"data shape: {data.shape}")
    random_shifts = load_distortions(dataset_name, distortion_type)

    model, temporal_contr_model = model_dict[dataset_name]
    
    embeddings = apply_embedding_function_shifts(data, lambda x: ts_tcc_embedding_function(x, model, temporal_contr_model, device), random_shifts)
    print(f"embeddings shape: {embeddings.shape}")
    path = f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/{distortion_type}/{dataset_name}.npy"
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (442, 427, 1)
embeddings shape: (442, 6, 100)
data shape: (500, 176, 1)
embeddings shape: (500, 6, 100)
data shape: (350, 463, 1)
embeddings shape: (350, 6, 100)
data shape: (500, 99, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 152, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 131, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 60, 1)
embeddings shape: (500, 6, 100)
data shape: (112, 350, 1)
embeddings shape: (112, 6, 100)
data shape: (200, 275, 1)
embeddings shape: (200, 6, 100)
data shape: (60, 470, 1)
embeddings shape: (60, 6, 100)
data shape: (200, 150, 1)
embeddings shape: (200, 6, 100)
data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (56, 286, 1)
embeddings shape: (56, 6, 100)
data shape: (60, 570, 1)
embeddings shape: (60, 6, 100)
data shape: (200, 96, 1)
embeddings shape: (200, 6, 100)


### Self-supervised

In [67]:
subdir = "TS-TCC"
os.chdir(os.path.join(project_root, subdir))

In [68]:
model_dict = {}

for dataset_name in dataset_names:
    model, temporal_contr_model = load_models(dataset_name, device, training_type="self_supervised")
    model_dict[dataset_name] = (model, temporal_contr_model)

In [70]:
subdir = ""
os.chdir(os.path.join(project_root, subdir))

In [71]:
distortion_type = "shrink"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    print(f"data shape: {data.shape}")
    random_shrinks = load_distortions(dataset_name, distortion_type)

    model, temporal_contr_model = model_dict[dataset_name]
    
    embeddings = apply_embedding_function_shrunk(data, lambda x: ts_tcc_embedding_function(x, model, temporal_contr_model, device), random_shrinks)
    print(f"embeddings shape: {embeddings.shape}")
    path = f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/{distortion_type}/{dataset_name}.npy"
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (442, 427)
embeddings shape: (442, 6, 100)
data shape: (500, 176)
embeddings shape: (500, 6, 100)
data shape: (350, 463)
embeddings shape: (350, 6, 100)
data shape: (500, 99)
embeddings shape: (500, 6, 100)
data shape: (500, 152)
embeddings shape: (500, 6, 100)
data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (500, 131)
embeddings shape: (500, 6, 100)
data shape: (500, 60)
embeddings shape: (500, 6, 100)
data shape: (112, 350)
embeddings shape: (112, 6, 100)
data shape: (200, 275)
embeddings shape: (200, 6, 100)
data shape: (60, 470)
embeddings shape: (60, 6, 100)
data shape: (200, 150)
embeddings shape: (200, 6, 100)
data shape: (500, 128)
embeddings shape: (500, 6, 100)
data shape: (56, 286)
embeddings shape: (56, 6, 100)
data shape: (60, 570)
embeddings shape: (60, 6, 100)
data shape: (200, 96)
embeddings shape: (200, 6, 100)


In [73]:
distortion_type = "shift"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    data = data[:, :, None]
    print(f"data shape: {data.shape}")
    random_shifts = load_distortions(dataset_name, distortion_type)

    model, temporal_contr_model = model_dict[dataset_name]
    
    embeddings = apply_embedding_function_shifts(data, lambda x: ts_tcc_embedding_function(x, model, temporal_contr_model, device), random_shifts)
    print(f"embeddings shape: {embeddings.shape}")
    path = f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/{distortion_type}/{dataset_name}.npy"
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (442, 427, 1)
embeddings shape: (442, 6, 100)
data shape: (500, 176, 1)
embeddings shape: (500, 6, 100)
data shape: (350, 463, 1)
embeddings shape: (350, 6, 100)
data shape: (500, 99, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 152, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 131, 1)
embeddings shape: (500, 6, 100)
data shape: (500, 60, 1)
embeddings shape: (500, 6, 100)
data shape: (112, 350, 1)
embeddings shape: (112, 6, 100)
data shape: (200, 275, 1)
embeddings shape: (200, 6, 100)
data shape: (60, 470, 1)
embeddings shape: (60, 6, 100)
data shape: (200, 150, 1)
embeddings shape: (200, 6, 100)
data shape: (500, 128, 1)
embeddings shape: (500, 6, 100)
data shape: (56, 286, 1)
embeddings shape: (56, 6, 100)
data shape: (60, 570, 1)
embeddings shape: (60, 6, 100)
data shape: (200, 96, 1)
embeddings shape: (200, 6, 100)
